# PKG Attrition — Source Profiling EDA · v3 (scale-first)

v2 failed on the first cell: an unfiltered `countDistinct` plus a timestamp
through Arrow, against the full staging table. That is a design problem, not a
bug — **the notebook must never scan raw staging more than once.**

### Architecture

```
staging (huge)  ──┐
neo4j_customer ───┼──> [ ONE PASS ] ──> legs_{month}.parquet   (ego-centric, narrow)
deposit book   ──┘                            │
                                              ├──> edges_monthly.parquet
                                              ├──> cust_monthly.parquet
                                              └──> cpty_dim.parquet
                                                       │
                                           every profiling stage reads THESE
```

**The one pass** projects ~14 columns, prunes by date, filters to corporate
customers in the deposit book, and explodes each transaction into ego legs. It
writes parquet partitioned by month. Everything downstream reads columnar,
partition-pruned parquet — typically one to two orders of magnitude less I/O
than the source.

### Rules this notebook follows

| Rule | Why |
|---|---|
| `PROFILE_MONTHS` starts at 3 | prove the pipeline cheap, then widen |
| `approx_count_distinct` everywhere | exact distinct is a full shuffle of the key |
| no `.cache()` on anything wide | cache on a huge frame spills or dies; parquet instead |
| every `.toPandas()` is on an aggregate | never a collect of row-level data |
| timestamps cast to string before Arrow | the v2 failure |
| rebuild is skipped if output exists | resumable, matching the pipeline convention |
| AQE skew join on | hub counterparties skew every `groupBy(cpty_key)` |

### Ego-leg model

One transaction becomes one leg per PNC customer involved. An internal
customer-to-customer payment produces **two** legs (one per side); an external
payment produces one. Each leg is `(ego, direction, counterparty, amount, month)`.
Counterparty namespaces are kept distinct: `PNC:<mdm_id>` for on-us,
`unq_cpty_acct_id` for off-us. This is the deposit-anchored extraction, and it is
the shape every downstream aggregation wants.

In [ ]:
# ============================================================================
# CONFIG
# ============================================================================
import os, re, json, time
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window as W

CONFIG = dict(
    TXN_TABLE  = "<db>.<staging_transactions>",
    DEP_TABLE  = "<db>.<deposit_wide>",
    CUST_TABLE = "<db>.neo4j_customer",

    # Working area. HDFS path preferred — these are cluster-sized outputs.
    WORK_DIR   = "hdfs:///user/<you>/pkg_attrition/work",
    OUT_DIR    = "../eda/attrition",     # small CSV results, local

    # *** START SMALL. Widen only after the full pipeline runs clean. ***
    PROFILE_MONTHS = ["2025-04", "2025-05", "2025-06"],

    # Population gate
    PARTY_TYPES    = ["O"],       # organisations only; the TM population
    REQUIRE_DEPOSIT = True,       # ego must exist in the deposit book

    # Force rebuild of the working tables even if they exist
    REBUILD        = False,

    # Sampling for format / name inspection only
    SAMPLE_FRAC    = 0.001,

    # Thresholds
    BALANCE_FLOOR     = 25_000,
    CURRENT_RULE_DROP = 0.30,
    SILENCE_MONTHS    = 3,
    HLL_RSD           = 0.01,     # approx_count_distinct relative error
)

T = dict(
    pays_id="mdm_id_pays", pays_name="customer_name_pays", pays_acct="pnc_dep_acct_pays",
    recv_id="mdm_id_receives", recv_name="customer_name_receives", recv_acct="pnc_dep_acct_receives",
    cpty_key="unq_cpty_acct_id", cpty_name="cpty_name", cpty_type="cpty_type",
    cpty_fi="cpty_fin_entity_name",
    txn_id="trans_id", amount="trans_amt", currency="trans_currency", date="trans_dt",
    rail="payment_rail", category="category", cat_prefix="category_prefix",
    src_syst="src_syst", np_key="np_key",
    merch_id="merchant_id", merch_cat="merchant_cat_cd", load_ts="hdfs_load_ts",
)
DEP  = dict(cust="cust_pwr_id", rltn="rltn_pwr_id", latest="latest_month",
            prior_avg="prior_avg", recent_avg="recent_avg", pct_change="pct_change",
            peak="peak_balance", latest_bal="latest_month_bal",
            n_acct="latest_num_accounts", n_closed="latest_num_closed_accounts")
CUST = dict(mdm_id="mdm_id", pwr_id="cust_pwr_id", party="party_type",
            naics="naics_cd", name="customer_name")

In [ ]:
# ============================================================================
# SPARK TUNING & HELPERS
# ============================================================================
spark = SparkSession.builder.getOrCreate()

# AQE does the work that manual partition tuning used to. skewJoin matters here
# specifically: hub counterparties (card networks, payroll processors) put
# hundreds of millions of rows behind a single key, and without it one task in
# every groupBy(cpty_key) runs until it dies.
for k, v in {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.adaptive.localShuffleReader.enabled": "true",
    "spark.sql.shuffle.partitions": "2000",
    "spark.sql.parquet.filterPushdown": "true",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    # The v2 failure: Arrow died mid-computation and there was no fallback.
    "spark.sql.execution.arrow.pyspark.fallback.enabled": "true",
    "spark.sql.autoBroadcastJoinThreshold": str(64 * 1024 * 1024),
    "spark.sql.sources.partitionOverwriteMode": "dynamic",
}.items():
    spark.conf.set(k, v)

OUT, WORK = CONFIG["OUT_DIR"], CONFIG["WORK_DIR"]
os.makedirs(OUT, exist_ok=True)
R, TIMINGS = {}, []


class timed:
    """Wall time per stage. A run that will overrun is visible early rather than
    at hour seven — same reason pkg_runtime exists."""
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); print(f"\n>>> {self.label}"); return self
    def __exit__(self, *a):
        dt = time.time() - self.t
        TIMINGS.append({"stage": self.label, "seconds": round(dt, 1)})
        print(f"<<< {self.label}: {dt/60:.1f} min")


def save(pdf, name, note=""):
    """Write a small result frame. Guards against an accidental large collect."""
    if len(pdf) > 5000:
        raise ValueError(f"{name}: {len(pdf)} rows is not a summary — aggregate further")
    pdf.to_csv(os.path.join(OUT, f"{name}.csv"), index=False)
    print(f"\n--- {name}{' · ' + note if note else ''}")
    with pd.option_context("display.max_rows", 80, "display.width", 220,
                           "display.float_format", lambda v: f"{v:,.4f}"):
        print(pdf.to_string(index=False))
    return pdf


def path(name):
    return f"{WORK}/{name}"


def exists(name):
    try:
        spark.read.parquet(path(name)).limit(1).count()
        return True
    except Exception:
        return False


def present(c):
    """Non-null and non-blank, collapsed once rather than by whichever fillna runs first."""
    col = F.col(c) if isinstance(c, str) else c
    return col.isNotNull() & (F.trim(col.cast("string")) != "")


def ndv(c, rsd=None):
    """Approximate distinct count. Exact countDistinct is a full shuffle of the key
    and is what killed v2 — at profiling precision the HLL answer is the same answer."""
    return F.approx_count_distinct(c, rsd or CONFIG["HLL_RSD"])


def norm_name(col):
    c = F.upper(F.trim(col))
    c = F.regexp_replace(c, r"[^A-Z0-9 ]", " ")
    c = F.regexp_replace(c, r"\b(LLC|L L C|INC|INCORPORATED|CORP|CORPORATION|CO|LP|LLP|"
                            r"PLLC|PC|LTD|LIMITED|DBA|THE)\b", " ")
    return F.trim(F.regexp_replace(c, r"\s+", " "))


def dshare(df, cols, name, note="", amt="amount"):
    g = (df.groupBy(*cols)
           .agg(F.count(F.lit(1)).alias("n_legs"), F.sum(amt).alias("dollars"))
           .toPandas())
    g["share_legs"] = g["n_legs"] / g["n_legs"].sum()
    g["share_dollars"] = g["dollars"] / g["dollars"].sum()
    return save(g.sort_values("dollars", ascending=False), name, note)


print("spark tuned ·", WORK)

---
# PHASE A — Build once

## A1 · Range discovery, without a full scan

`min`/`max` on the partition column is metadata-cheap if the table is partitioned
on it and a full scan if it is not. `SHOW PARTITIONS` is tried first and costs
nothing. If staging is not partitioned by date, that fact is itself a finding —
every query in this programme will be a full scan until it is.

In [ ]:
with timed("A1 range discovery"):
    try:
        parts = spark.sql(f"SHOW PARTITIONS {CONFIG['TXN_TABLE']}").toPandas()
        print(f"PARTITIONED · {len(parts)} partitions")
        print(parts.head(5).to_string(index=False))
        print(parts.tail(5).to_string(index=False))
        R["partitioned"] = True
    except Exception as e:
        print("NOT PARTITIONED (or no access to partition metadata).")
        print("*** Every read below is a full scan. Raise this — it gates the whole programme. ***")
        R["partitioned"] = False

    print("\nschema:")
    print(pd.DataFrame(spark.table(CONFIG["TXN_TABLE"]).dtypes,
                       columns=["column", "dtype"]).to_string(index=False))

## A2 · The corporate key set

Small by construction: organisations with a `cust_pwr_id` that appear in the
deposit book. Written once and broadcast into the one pass, so staging is
filtered on the way in rather than scanned and discarded.

In [ ]:
with timed("A2 corporate key set"):
    if CONFIG["REBUILD"] or not exists("corp_keys"):
        cust = spark.table(CONFIG["CUST_TABLE"])
        corp = (cust
            .filter(F.col(CUST["party"]).isin(CONFIG["PARTY_TYPES"]) & present(CUST["pwr_id"]))
            .select(F.col(CUST["mdm_id"]).alias("mdm_id"),
                    F.col(CUST["pwr_id"]).alias("cust_pwr_id"),
                    F.col(CUST["naics"]).alias("naics_cd"),
                    norm_name(F.col(CUST["name"])).alias("ego_name_norm"))
            .dropDuplicates(["mdm_id"]))

        if CONFIG["REQUIRE_DEPOSIT"]:
            dep_ids = (spark.table(CONFIG["DEP_TABLE"])
                            .select(F.col(DEP["cust"]).alias("cust_pwr_id")).distinct())
            corp = corp.join(F.broadcast(dep_ids), "cust_pwr_id")

        corp.write.mode("overwrite").parquet(path("corp_keys"))

    corp = spark.read.parquet(path("corp_keys"))
    n_corp = corp.count()
    print(f"corporate egos: {n_corp:,}")
    R["n_corporate_egos"] = n_corp

    # 1:1 verification. If mdm_id fans out per cust_pwr_id, one "customer" in the
    # deposit table is several graph nodes and every per-customer aggregate is
    # wrong by an unknown factor.
    save(corp.groupBy("cust_pwr_id").agg(F.count(F.lit(1)).alias("n_mdm"))
             .groupBy("n_mdm").count().orderBy("n_mdm").limit(20).toPandas(),
         "A2_mdm_per_pwr_id", "*** anything above n_mdm=1 breaks the 1:1 assumption ***")

## A3 · The one pass — ego legs

**The only time raw staging is read.** Projects 14 columns, prunes to
`PROFILE_MONTHS`, filters to corporate egos, explodes to legs, writes parquet
partitioned by month.

The counterparty of a leg is the *other* side: another PNC customer when the
opposite MDM id is present (`PNC:` prefixed so the namespaces cannot collide),
otherwise `unq_cpty_acct_id`. `cpty_key_src` records which, because on-us and
off-us counterparties are not interchangeable evidence and collapsing them is
the same class of error as collapsing observed and inferred NAICS.

In [ ]:
with timed("A3 build ego legs — THE ONE PASS"):
    if CONFIG["REBUILD"] or not exists("legs"):
        raw = (spark.table(CONFIG["TXN_TABLE"])
               .withColumn("month", F.substring(F.col(T["date"]), 1, 7))
               .filter(F.col("month").isin(CONFIG["PROFILE_MONTHS"]))
               .select(
                   "month",
                   F.col(T["txn_id"]).alias("txn_id"),
                   F.col(T["date"]).alias("trans_dt"),
                   F.col(T["amount"]).cast("double").alias("amount"),
                   F.col(T["currency"]).alias("currency"),
                   F.col(T["rail"]).alias("rail"),
                   F.col(T["category"]).alias("category"),
                   F.col(T["cat_prefix"]).alias("cat_prefix"),
                   F.col(T["src_syst"]).alias("src_syst"),
                   F.col(T["pays_id"]).alias("pays_id"), F.col(T["pays_name"]).alias("pays_name"),
                   F.col(T["pays_acct"]).alias("pays_acct"),
                   F.col(T["recv_id"]).alias("recv_id"), F.col(T["recv_name"]).alias("recv_name"),
                   F.col(T["recv_acct"]).alias("recv_acct"),
                   F.col(T["cpty_key"]).alias("cpty_raw"), F.col(T["cpty_name"]).alias("cpty_name"),
                   F.col(T["cpty_type"]).alias("cpty_type"), F.col(T["cpty_fi"]).alias("cpty_fi"),
                   F.col(T["merch_id"]).alias("merch_id"),
               ))

        # Topology from the null pattern — there is no direction column.
        raw = (raw
            .withColumn("_p", present("pays_id")).withColumn("_r", present("recv_id"))
            .withColumn("topology",
                F.when(F.col("_p") & F.col("_r"), "INTERNAL_C2C")
                 .when(~F.col("_p") & F.col("_r"), "INBOUND")
                 .when(F.col("_p") & ~F.col("_r"), "OUTBOUND")
                 .otherwise("ORPHAN"))
            .withColumn("is_self_loop", F.col("_p") & F.col("_r") & (F.col("pays_id") == F.col("recv_id"))))

        # Explode into legs. OUT leg exists when the payer is a PNC customer;
        # IN leg when the receiver is. INTERNAL_C2C therefore yields two.
        leg_out = F.when(F.col("_p"), F.struct(
            F.lit("OUT").alias("ego_dir"),
            F.col("pays_id").alias("ego_mdm"), F.col("pays_name").alias("ego_name"),
            F.col("pays_acct").alias("ego_acct"),
            F.coalesce(F.concat(F.lit("PNC:"), F.col("recv_id")), F.col("cpty_raw")).alias("cpty_key"),
            F.when(F.col("_r"), F.lit("on_us")).otherwise(F.lit("off_us")).alias("cpty_key_src"),
            F.coalesce(F.col("recv_name"), F.col("cpty_name")).alias("cpty_name_eff")))
        leg_in = F.when(F.col("_r"), F.struct(
            F.lit("IN").alias("ego_dir"),
            F.col("recv_id").alias("ego_mdm"), F.col("recv_name").alias("ego_name"),
            F.col("recv_acct").alias("ego_acct"),
            F.coalesce(F.concat(F.lit("PNC:"), F.col("pays_id")), F.col("cpty_raw")).alias("cpty_key"),
            F.when(F.col("_p"), F.lit("on_us")).otherwise(F.lit("off_us")).alias("cpty_key_src"),
            F.coalesce(F.col("pays_name"), F.col("cpty_name")).alias("cpty_name_eff")))

        legs = (raw
            .withColumn("leg", F.explode(F.array(leg_out, leg_in)))
            .filter(F.col("leg").isNotNull())
            .select("month", "txn_id", "trans_dt", "amount", "currency", "rail", "category",
                    "cat_prefix", "src_syst", "topology", "is_self_loop",
                    "cpty_raw", "cpty_name", "cpty_type", "cpty_fi", "merch_id",
                    F.col("leg.ego_dir").alias("ego_dir"),
                    F.col("leg.ego_mdm").alias("mdm_id"),
                    F.col("leg.ego_name").alias("ego_name"),
                    F.col("leg.ego_acct").alias("ego_acct"),
                    F.col("leg.cpty_key").alias("cpty_key"),
                    F.col("leg.cpty_key_src").alias("cpty_key_src"),
                    F.col("leg.cpty_name_eff").alias("cpty_name_eff"))
            # Population gate applied HERE, inside the one pass, so the corporate
            # filter reduces the shuffle rather than running after it.
            .join(F.broadcast(corp.select("mdm_id", "cust_pwr_id", "naics_cd", "ego_name_norm")),
                  "mdm_id"))

        (legs.write.mode("overwrite").partitionBy("month").parquet(path("legs")))

    legs = spark.read.parquet(path("legs"))
    print("legs written ·", path("legs"))

## A4 · Working-table sanity

Everything from here reads `legs`. Three facts to establish before trusting any
aggregate: how many legs survived the gate, whether `txn_id` duplicates within a
month (which would mean the source double-books before we even explode), and the
currency and sign profile.

In [ ]:
with timed("A4 sanity"):
    one = CONFIG["PROFILE_MONTHS"][-1]
    s = (legs.filter(F.col("month") == one).agg(
            F.count(F.lit(1)).alias("n_legs"),
            ndv("txn_id").alias("n_txn_approx"),
            ndv("cust_pwr_id").alias("n_egos_approx"),
            ndv("cpty_key").alias("n_cpty_approx"),
            F.sum("amount").alias("dollars"),
            F.sum(F.when(F.col("amount") < 0, 1).otherwise(0)).alias("n_negative"),
            F.sum(F.when(F.col("amount") == 0, 1).otherwise(0)).alias("n_zero"),
         ).toPandas())
    s["legs_per_txn"] = s["n_legs"] / s["n_txn_approx"]
    save(s.T.reset_index().rename(columns={"index": "metric", 0: "value"}),
         "A4_working_shape", f"one month ({one}); legs_per_txn ~1.x is expected")
    R["legs_per_txn"] = float(s["legs_per_txn"].iloc[0])

    save(legs.groupBy("month").agg(
            F.count(F.lit(1)).alias("n_legs"),
            ndv("cust_pwr_id").alias("n_egos"),
            F.sum("amount").alias("dollars")).orderBy("month").toPandas(),
         "A4_monthly_volume", "a ramp at either end is an ingestion artefact, not a trend")

    save(legs.groupBy("currency").agg(F.count(F.lit(1)).alias("n"),
                                      F.sum("amount").alias("dollars")).toPandas(),
         "A4_currency_mix", "non-USD needs an explicit policy before anything is summed")

---
# PHASE B — Profile the working table

## B1 · Topology

`OUTBOUND` is the half the graph has never had — everything the brief calls
Group A lives there. `INTERNAL_C2C` is the ~12%-coverage world the prior study
was confined to. `ORPHAN` should be ~0 and is a defect if it is not.

Self-loops are one customer moving between its own PNC accounts. Not payments,
so they must not enter flow metrics — but they are the only on-us analogue of the
off-us self-payment we want to detect, so they are counted, not dropped.

In [ ]:
with timed("B1 topology"):
    dshare(legs, ["topology"], "B1_topology_mix",
           "*** OUTBOUND is the new half. ORPHAN > ~0 is a defect. ***")
    dshare(legs, ["ego_dir", "cpty_key_src"], "B1_direction_x_scope",
           "on_us vs off_us counterparties are not interchangeable evidence")
    dshare(legs, ["topology", "rail"], "B1_topology_x_rail")

    save(legs.filter("is_self_loop").agg(
            F.count(F.lit(1)).alias("n"), ndv("cust_pwr_id").alias("n_customers"),
            F.sum("amount").alias("dollars")).toPandas(),
         "B1_self_loops", "exclude from flow; keep as the on-us self-payment analogue")

    tot = legs.agg(F.sum("amount")).first()[0] or 1
    out = legs.filter("ego_dir='OUT' AND cpty_key_src='off_us'").agg(F.sum("amount")).first()[0] or 0
    R["off_us_outbound_dollar_share"] = float(out) / float(tot)

## B2 · Rail and product taxonomy

Five overlapping descriptors. The rail metrics spec has been blocked on knowing
which is the real taxonomy and how internal book transfers are represented. The
cross-tab settles it instead of picking one on faith.

In [ ]:
with timed("B2 taxonomy"):
    for c in ["rail", "category", "cat_prefix", "src_syst"]:
        dshare(legs, [c], f"B2_{c}")
    dshare(legs, ["rail", "category"], "B2_rail_x_category",
           "is category a refinement of rail, or an orthogonal axis?")
    save(legs.groupBy("rail").agg(
            F.count(F.lit(1)).alias("n"),
            F.expr("percentile_approx(amount, 0.5)").alias("p50"),
            F.expr("percentile_approx(amount, 0.95)").alias("p95"),
            F.expr("percentile_approx(amount, 0.999)").alias("p999")).toPandas(),
         "B2_amount_by_rail",
         "card sits orders of magnitude below wire — every dollar rule must be rail-relative")

## B3 · Counterparty identifiability — the Group A gate

`unq_cpty_acct_id` is built upstream as `rn + account_id`, **falling back to the
name** when the account is absent. Those are not the same object:

- **account-derived** → a stable node. Appearing and disappearing is a real event.
- **name-derived** → a node whose identity is a string. Two spellings are two
  nodes; a spelling change reads as a lost relationship, which lands directly on
  Group B's relationship-dissolution metrics.

The provenance regex is a guess. The sample listing below is there to falsify it
before any share computed from it is believed.

In [ ]:
with timed("B3 identifiability"):
    ext = (legs.filter("cpty_key_src='off_us'")
        .withColumn("_key", present("cpty_key"))
        .withColumn("_name", present("cpty_name_eff"))
        .withColumn("_fi", present("cpty_fi"))
        .withColumn("key_provenance",
            F.when(~present("cpty_key"), "none")
             .when(F.col("cpty_key").rlike(r"^[0-9][0-9\-_|]*[0-9]$"), "account_derived")
             .otherwise("name_derived")))

    dshare(ext, ["key_provenance"], "B3_key_provenance",
           "*** account_derived = stable node; name_derived = a string ***")
    dshare(ext, ["ego_dir", "key_provenance"], "B3_provenance_by_direction")
    dshare(ext, ["rail", "key_provenance"], "B3_provenance_by_rail")

    save(ext.filter(present("cpty_key")).select("cpty_key", "key_provenance", "rail")
            .sample(CONFIG["SAMPLE_FRAC"]).limit(40).toPandas(),
         "B3_key_format_sample", "*** READ THIS — confirm the provenance regex ***")

    ext = ext.withColumn("id_class",
        F.when(F.col("_key") & F.col("_name") & F.col("_fi"), "key+name+fi")
         .when(F.col("_key") & F.col("_name"), "key+name")
         .when(F.col("_key"), "key_only")
         .when(F.col("_name"), "name_only").otherwise("unidentifiable"))
    dshare(ext, ["id_class"], "B3_identifiability", "*** THE Q3 ANSWER, dollar-weighted ***")
    dshare(ext, ["rail", "id_class"], "B3_identifiability_by_rail",
           "coverage is rail-biased; rail mix correlates with industry and size")
    dshare(ext, ["cpty_type"], "B3_cpty_type", "does this already carry entity typing?")

    et = ext.agg(F.sum("amount").alias("d"),
                 F.sum(F.when(F.col("_name"), F.col("amount")).otherwise(0.0)).alias("dn"),
                 F.sum(F.when(F.col("_key"), F.col("amount")).otherwise(0.0)).alias("dk")).first()
    R["cpty_name_dollar_coverage"] = float(et["dn"]) / float(et["d"] or 1)
    R["cpty_key_dollar_coverage"]  = float(et["dk"]) / float(et["d"] or 1)

## B4 · `cpty_fin_entity_name` — the FI registry, already built

This column removes the §12 open item entirely. `fi_destination_flag` becomes a
lookup, and the outbound destination ranking is a competitive map.

Check whether PNC itself appears: those rows are on-us and must not count as
off-us flow.

In [ ]:
with timed("B4 FI registry"):
    save(ext.groupBy("ego_dir").agg(
            F.mean(F.col("_fi").cast("double")).alias("fi_populated_legs"),
            (F.sum(F.when(F.col("_fi"), F.col("amount")).otherwise(0.0))
             / F.sum("amount")).alias("fi_populated_dollars")).toPandas(),
         "B4_fi_coverage", "*** replaces the FI-name-list open item ***")

    save(ext.filter(F.col("_fi")).groupBy("cpty_fi")
            .agg(F.count(F.lit(1)).alias("n"),
                 ndv("cust_pwr_id").alias("n_customers"),
                 F.sum("amount").alias("dollars"))
            .orderBy(F.desc("dollars")).limit(100).toPandas(),
         "B4_top_financial_entities", "*** the competitive map. Is PNC in this list? ***")

    save(ext.filter("ego_dir='OUT'").filter(F.col("_fi")).groupBy("cpty_fi")
            .agg(ndv("cust_pwr_id").alias("n_customers"), F.sum("amount").alias("dollars"))
            .orderBy(F.desc("dollars")).limit(60).toPandas(),
         "B4_outbound_destination_banks", "where money leaves to")

In [ ]:
with timed("B4b hub seed"):
    # Payroll processors, card networks and PNC book-transfer accounts should all
    # appear here — and each needs a DIFFERENT policy. A flat exclusion list
    # collapses signal hubs, noise hubs and internal accounts into one bucket.
    save(ext.filter(F.col("_name")).groupBy(norm_name(F.col("cpty_name_eff")).alias("norm_name"))
            .agg(F.count(F.lit(1)).alias("n"),
                 ndv("cust_pwr_id").alias("n_customers"),
                 F.sum("amount").alias("dollars"),
                 F.first("cpty_fi", ignorenulls=True).alias("fi_sample"),
                 F.first("cpty_type", ignorenulls=True).alias("type_sample"))
            .orderBy(F.desc("n_customers")).limit(100).toPandas(),
         "B4_top_counterparty_names", "hub taxonomy seed: payroll / network / internal / anchor")

    # Merchant rows are a second counterparty namespace. Whether they also populate
    # cpty_key decides if card flow joins the same graph or sits beside it. Either
    # is workable; not knowing produces a graph where some card spend is an edge.
    save(legs.withColumn("_m", present("merch_id")).withColumn("_c", present("cpty_key"))
             .groupBy("_m", "_c").agg(F.count(F.lit(1)).alias("n"),
                                      F.sum("amount").alias("dollars")).toPandas(),
         "B4_merchant_vs_cpty", "*** merchant-only legs are a separate namespace ***")

---
# PHASE C — Derived tables

Three parquet outputs that every later stage and every downstream module reads.
Built from `legs`, never from staging.

In [ ]:
with timed("C1 edges_monthly"):
    if CONFIG["REBUILD"] or not exists("edges_monthly"):
        (legs.filter("NOT is_self_loop")
             .groupBy("cust_pwr_id", "cpty_key", "cpty_key_src", "ego_dir", "month")
             .agg(F.sum("amount").alias("amount"),
                  F.count(F.lit(1)).alias("volume"),
                  F.countDistinct("rail").alias("n_rails"),
                  F.min("trans_dt").alias("first_dt"),
                  F.max("trans_dt").alias("last_dt"),
                  F.first("cpty_fi", ignorenulls=True).alias("cpty_fi"),
                  F.first("cpty_name_eff", ignorenulls=True).alias("cpty_name"))
             .write.mode("overwrite").partitionBy("month").parquet(path("edges_monthly")))
    edges = spark.read.parquet(path("edges_monthly"))
    print("edges:", path("edges_monthly"))

In [ ]:
with timed("C2 cust_monthly"):
    if CONFIG["REBUILD"] or not exists("cust_monthly"):
        (legs.filter("NOT is_self_loop")
             .groupBy("cust_pwr_id", "month")
             .agg(F.sum(F.when(F.col("ego_dir") == "OUT", F.col("amount")).otherwise(0.0)).alias("gross_out"),
                  F.sum(F.when(F.col("ego_dir") == "IN",  F.col("amount")).otherwise(0.0)).alias("gross_in"),
                  F.sum(F.when((F.col("ego_dir") == "OUT") & (F.col("cpty_key_src") == "off_us"),
                               F.col("amount")).otherwise(0.0)).alias("off_us_out"),
                  F.count(F.lit(1)).alias("n_legs"),
                  ndv("cpty_key").alias("n_cpty"),
                  ndv(F.when(F.col("ego_dir") == "OUT", F.col("cpty_key"))).alias("n_cpty_out"),
                  ndv("ego_acct").alias("n_active_accts"))
             .withColumn("net_flow", F.col("gross_in") - F.col("gross_out"))
             .withColumn("off_us_out_share", F.col("off_us_out") / F.greatest(F.col("gross_out"), F.lit(1.0)))
             .write.mode("overwrite").partitionBy("month").parquet(path("cust_monthly")))
    cm = spark.read.parquet(path("cust_monthly"))
    print("cust_monthly:", path("cust_monthly"))

In [ ]:
with timed("C3 cpty_dim"):
    if CONFIG["REBUILD"] or not exists("cpty_dim"):
        # Two-stage reduce. Distinct (cpty, ego) pairs first, then count — this is
        # what keeps a hub with hundreds of millions of legs from landing on one task.
        pairs = legs.filter("cpty_key_src='off_us'").select("cpty_key", "cust_pwr_id").distinct()
        attrs = (legs.filter("cpty_key_src='off_us'")
                 .groupBy("cpty_key")
                 .agg(F.first("cpty_name_eff", ignorenulls=True).alias("cpty_name"),
                      F.first("cpty_fi", ignorenulls=True).alias("cpty_fi"),
                      F.first("cpty_type", ignorenulls=True).alias("cpty_type"),
                      F.sum("amount").alias("dollars"),
                      F.count(F.lit(1)).alias("n_legs"),
                      F.min("month").alias("first_month"),
                      F.max("month").alias("last_month")))
        (pairs.groupBy("cpty_key").agg(F.count(F.lit(1)).alias("n_customers"))
              .join(attrs, "cpty_key")
              .write.mode("overwrite").parquet(path("cpty_dim")))
    cpd = spark.read.parquet(path("cpty_dim"))

    # Fan-in bounds Group C: a counterparty seen by ONE customer has no independent
    # health signal, so lost_cpty_health_index is undefined for it.
    save(cpd.agg(F.count(F.lit(1)).alias("n_counterparties"),
                 F.mean((F.col("n_customers") >= 2).cast("double")).alias("share_2plus"),
                 F.mean((F.col("n_customers") >= 5).cast("double")).alias("share_5plus"),
                 F.max("n_customers").alias("max_customers")).toPandas(),
         "C3_cpty_fanin", "*** share_5plus bounds Group C coverage ***")

    save(edges.filter("ego_dir='OUT'").groupBy("cust_pwr_id", "month")
              .agg(F.count(F.lit(1)).alias("n_cpty"))
              .agg(F.mean("n_cpty").alias("mean"),
                   F.expr("percentile_approx(n_cpty, 0.5)").alias("p50"),
                   F.expr("percentile_approx(n_cpty, 0.95)").alias("p95"),
                   F.expr("percentile_approx(n_cpty, 0.999)").alias("p999"),
                   F.max("n_cpty").alias("max")).toPandas(),
         "C3_outbound_fanout", "sizes the Tier-1 monthly graph")

---
# PHASE D — Deposit panel

Small: one row per `cust_pwr_id`, months as columns, **monthly average balance**.
Unpivoted here. The null pattern in `bal_*` carries information no other column
holds — **trailing nulls are a candidate departure label**, dated, available today,
and not derived from a threshold on a declining balance.

In [ ]:
with timed("D1 deposit unpivot"):
    dep_w = spark.table(CONFIG["DEP_TABLE"])
    bal_cols = sorted([c for c in dep_w.columns if re.match(r"bal_\d{4}_\d{2}$", c)])
    print(f"{len(bal_cols)} balance columns: {bal_cols[0]} .. {bal_cols[-1]}")

    if CONFIG["REBUILD"] or not exists("dep_long"):
        pairs_sql = ", ".join([f"'{c[4:].replace('_','-')}', cast({c} as double)" for c in bal_cols])
        (dep_w.select(F.col(DEP["cust"]).alias("cust_pwr_id"),
                      F.expr(f"stack({len(bal_cols)}, {pairs_sql}) as (month, balance)"))
              .write.mode("overwrite").parquet(path("dep_long")))
    dep = spark.read.parquet(path("dep_long"))

    save(dep_w.agg(F.count(F.lit(1)).alias("n_rows"),
                   ndv(DEP["cust"]).alias("n_customers"),
                   ndv(DEP["rltn"]).alias("n_relationships"),
                   F.mean(F.col(DEP["n_acct"]).cast("double")).alias("mean_accounts"),
                   F.mean((F.col(DEP["n_closed"]) > 0).cast("double")).alias("share_any_closed")).toPandas(),
         "D1_deposit_shape", "n_rows == n_customers confirms one row per customer")

    save(dep.groupBy("month").agg(
            F.count(F.lit(1)).alias("n"),
            F.mean(F.col("balance").isNotNull().cast("double")).alias("share_non_null"),
            F.expr("percentile_approx(balance, 0.5)").alias("p50")).orderBy("month").toPandas(),
         "D1_month_coverage")

In [ ]:
with timed("D2 balance-series shapes"):
    pat = (dep.withColumn("live", F.col("balance").isNotNull() & (F.col("balance") != 0))
        .groupBy("cust_pwr_id")
        .agg(F.sum(F.col("live").cast("int")).alias("n_months_live"),
             F.min(F.when(F.col("live"), F.col("month"))).alias("first_live"),
             F.max(F.when(F.col("live"), F.col("month"))).alias("last_live"),
             F.max("month").alias("panel_end"))
        .withColumn("shape",
            F.when(F.col("n_months_live") == 0, "never_live")
             .when(F.col("last_live") == F.col("panel_end"), "live_at_end")
             .otherwise("STOPPED_BEFORE_END")))
    pat.write.mode("overwrite").parquet(path("dep_shape"))
    pat = spark.read.parquet(path("dep_shape"))

    save(pat.groupBy("shape").count().toPandas(), "D2_series_shapes",
         "*** STOPPED_BEFORE_END is a departure label available TODAY ***")
    save(pat.filter("shape='STOPPED_BEFORE_END'").groupBy("last_live").count()
            .orderBy("last_live").toPandas(),
         "D2_stop_month", "flat = real churn; a spike at one month = artefact")
    R["n_stopped_before_end"] = int(pat.filter("shape='STOPPED_BEFORE_END'").count())

## D3 · Coverage — the number that replaces ~12%

The prior verdict (graph as coincident, not leading) was explicitly conditional
on ~12% deposit-book visibility. Deposit-anchored extraction should push this
near-total. If it does not, the reason must be found before anything is built on it.

In [ ]:
with timed("D3 coverage"):
    dep_ids = dep_w.select(F.col(DEP["cust"]).alias("cust_pwr_id")).distinct()
    seen    = cm.select("cust_pwr_id").distinct().withColumn("_any", F.lit(1))
    seen_o  = (edges.filter("ego_dir='OUT' AND cpty_key_src='off_us'")
                    .select("cust_pwr_id").distinct().withColumn("_out", F.lit(1)))

    cov = (dep_ids.join(seen, "cust_pwr_id", "left").join(seen_o, "cust_pwr_id", "left")
        .agg(F.count(F.lit(1)).alias("n_deposit_customers"),
             F.sum(F.coalesce("_any", F.lit(0))).alias("n_any_txn"),
             F.sum(F.coalesce("_out", F.lit(0))).alias("n_off_us_outbound")).toPandas())
    cov["coverage_any"] = cov["n_any_txn"] / cov["n_deposit_customers"]
    cov["coverage_outbound"] = cov["n_off_us_outbound"] / cov["n_deposit_customers"]
    save(cov, "D3_coverage",
         "*** replaces ~12%. coverage_outbound is the Group A population. ***")
    R["deposit_txn_coverage"] = float(cov["coverage_any"].iloc[0])
    R["deposit_outbound_coverage"] = float(cov["coverage_outbound"].iloc[0])
    # NOTE: over PROFILE_MONTHS only. A customer inactive in these three months is
    # not invisible — it is quiet. Re-read this after widening the window.

## D4 · Does staging explain balance movement?

**Exact reconciliation is impossible** — the balance is a monthly *average*, so
its delta is a smoothed function of within-month flow, not the sum of it. Any
residual conflates the averaging with missing data.

What is measurable: explained variance, and customer-months where the balance
moved with **zero** staging activity. That second share is a hard ceiling on
anything transaction-derived.

In [ ]:
with timed("D4 flow vs balance"):
    wd = W.partitionBy("cust_pwr_id").orderBy("month")
    panel = (dep.filter(F.col("month").isin(CONFIG["PROFILE_MONTHS"]))
        .withColumn("bal_prev", F.lag("balance").over(wd))
        .withColumn("bal_delta", F.col("balance") - F.col("bal_prev"))
        .filter(F.col("bal_prev").isNotNull() & F.col("balance").isNotNull())
        .join(cm, ["cust_pwr_id", "month"], "left")
        .fillna({"net_flow": 0.0, "gross_out": 0.0, "n_legs": 0}))

    save(panel.agg(F.count(F.lit(1)).alias("n_customer_months"),
                   F.corr("bal_delta", "net_flow").alias("corr_delta_netflow"),
                   F.corr("balance", "gross_out").alias("corr_level_grossout")).toPandas(),
         "D4_flow_vs_balance",
         "*** corr near 1 => ONE ledger; identity is the new axis, not amount ***")
    R["corr_delta_netflow"] = float(
        panel.agg(F.corr("bal_delta", "net_flow")).first()[0] or 0)

    save(panel.withColumn("bucket",
            F.when((F.col("n_legs") == 0) & (F.abs(F.col("bal_delta")) > 1000), "NO_TXN_BUT_BALANCE_MOVED")
             .when(F.col("n_legs") == 0, "no_txn_flat")
             .otherwise("has_txn"))
         .groupBy("bucket").agg(F.count(F.lit(1)).alias("n"),
             F.expr("percentile_approx(abs(bal_delta), 0.5)").alias("p50_abs_delta")).toPandas(),
         "D4_unexplained_movement",
         "*** a hard ceiling on transaction-derived features ***")

---
# PHASE E — Labels, base rates, power

## E1 · Same-name outflow

The brief's priority signal. Two things must be measured before it can be used.

**The base rate.** Most corporates permanently maintain multiple bank
relationships. If a large share always shows same-name outflow, the flag as
specified fires constantly. The signal is a **new** destination or a step change
in share — a delta, not a level.

**The false-positive rate of exact matching**, estimable without any labelling:
`INTERNAL_C2C` legs where the two MDM ids differ but normalised names match are,
by construction, distinct legal entities sharing a name.

In [ ]:
with timed("E1 same-name"):
    fp = (legs.filter("topology='INTERNAL_C2C' AND NOT is_self_loop AND ego_dir='OUT'")
        .withColumn("nm", norm_name(F.col("ego_name")) == norm_name(F.col("cpty_name_eff")))
        .agg(F.count(F.lit(1)).alias("n_distinct_entity_pairs"),
             F.sum(F.col("nm").cast("int")).alias("n_false_matches"),
             F.mean(F.col("nm").cast("double")).alias("false_positive_rate")).toPandas())
    save(fp, "E1_name_match_false_positives",
         "*** the FP rate exact matching inherits ***")

    sn = (legs.filter("ego_dir='OUT' AND cpty_key_src='off_us'")
        .filter(present("cpty_name_eff"))
        .withColumn("same_name", norm_name(F.col("cpty_name_eff")) == F.col("ego_name_norm")))

    save(sn.groupBy("month").agg(
            ndv(F.when(F.col("same_name"), F.col("cust_pwr_id"))).alias("n_same_name"),
            ndv("cust_pwr_id").alias("n_total"),
            (F.sum(F.when(F.col("same_name"), F.col("amount")).otherwise(0.0))
             / F.sum("amount")).alias("dollar_share"))
         .withColumn("base_rate", F.col("n_same_name") / F.col("n_total"))
         .orderBy("month").toPandas(),
         "E1_same_name_base_rate", "*** high and flat => measure the DELTA ***")

    save(sn.filter("same_name").groupBy("cpty_fi")
           .agg(ndv("cust_pwr_id").alias("n_customers"), F.sum("amount").alias("dollars"))
           .orderBy(F.desc("n_customers")).limit(40).toPandas(),
         "E1_same_name_destination_banks", "where customers hold their other accounts")

## E2 · Account activity as a closure proxy

`latest_num_closed_accounts` is current-state only: it cannot date a closure, so
using it as a time-varying label would leak the outcome into every prior month.
**A monthly version is the pinned ask to the deposit team.**

Meanwhile `ego_acct` gives a per-month count of accounts that *transacted*. An
account going permanently silent is a dated behavioural closure, available today.
This validates the proxy against the one closure fact that exists.

In [ ]:
with timed("E2 closure proxy"):
    agree = (cm.groupBy("cust_pwr_id").agg(F.max("n_active_accts").alias("txn_max_accts"))
        .join(dep_w.select(F.col(DEP["cust"]).alias("cust_pwr_id"),
                           F.col(DEP["n_acct"]).cast("int").alias("dep_n_accts"),
                           (F.col(DEP["n_closed"]) > 0).alias("has_closed")), "cust_pwr_id"))
    save(agree.agg(F.corr("txn_max_accts", "dep_n_accts").alias("corr"),
                   F.mean((F.col("txn_max_accts") == F.col("dep_n_accts")).cast("double")).alias("share_exact"),
                   F.mean(F.col("txn_max_accts").cast("double")).alias("mean_txn"),
                   F.mean(F.col("dep_n_accts").cast("double")).alias("mean_dep")).toPandas(),
         "E2_account_count_agreement",
         "*** if these agree, account silence is a usable dated closure proxy ***")

## E3 · Episode funnel and power

The 30% rule on the unpivoted panel, then §6.2's exclusions, then graph
visibility. **This decides whether Steps 3–6 of the brief are worth running** —
25 metrics against matched controls at two horizons needs thousands of episodes.

In [ ]:
with timed("E3 episode funnel"):
    w3 = W.partitionBy("cust_pwr_id").orderBy("month")
    ep = (dep.filter(F.col("balance").isNotNull())
        .withColumn("avg3",  F.avg("balance").over(w3.rowsBetween(-2, 0)))
        .withColumn("avg6p", F.avg("balance").over(w3.rowsBetween(-8, -3)))
        .withColumn("n_hist", F.count("balance").over(w3.rowsBetween(-11, 0)))
        .withColumn("fwd3",  F.avg("balance").over(w3.rowsBetween(1, 3)))
        .withColumn("flag",  F.col("avg3") <= (1 - CONFIG["CURRENT_RULE_DROP"]) * F.col("avg6p")))
    ep.write.mode("overwrite").parquet(path("episodes"))
    ep = spark.read.parquet(path("episodes"))

    rows = []
    conds = [
        ("1_raw_30pct_rule",     F.lit(True)),
        ("2_plus_12mo_history",  F.col("n_hist") >= 12),
        ("3_plus_balance_floor", (F.col("n_hist") >= 12) & (F.col("avg6p") >= CONFIG["BALANCE_FLOOR"])),
        ("4_plus_persistence",   (F.col("n_hist") >= 12) & (F.col("avg6p") >= CONFIG["BALANCE_FLOOR"])
                                 & (F.col("fwd3") <= 1.1 * F.col("avg3"))),
    ]
    for label, cond in conds:
        rows.append({"step": label,
                     "n_customers": ep.filter(F.col("flag") & cond)
                                      .select("cust_pwr_id").distinct().count()})
    save(pd.DataFrame(rows), "E3_episode_funnel", "*** the last row is the real sample size ***")

    survivors = ep.filter(F.col("flag") & conds[-1][1]).select("cust_pwr_id").distinct()
    gv = (survivors.join(seen_o, "cust_pwr_id", "left")
        .agg(F.count(F.lit(1)).alias("n_episodes"),
             F.sum(F.coalesce("_out", F.lit(0))).alias("n_outbound_visible")).toPandas())
    gv["coverage"] = gv["n_outbound_visible"] / gv["n_episodes"]
    save(gv, "E3_episodes_graph_visible", "*** the population Group A can be tested on ***")
    R["episodes_outbound_visible"] = int(gv["n_outbound_visible"].iloc[0])

## E4 · Candidate labels

No status field exists. Three targets compared so the choice is made on evidence:

| Label | Source | Circular? |
|---|---|---|
| `balance_30pct` | the current rule | **yes** — deposit-defined |
| `series_stopped` | trailing nulls in `bal_*` | no |
| `txn_silence` | N months with no staging activity | **no** — independent of deposits |

`txn_silence` breaks the circularity. A deposit model predicting a deposit-derived
event wins by construction; a behaviour-defined target is a fair test of whether
payment structure leads. If the agreement matrix shows these near-identical the
choice is moot — if not, it is the most consequential decision in the study.

In [ ]:
with timed("E4 labels"):
    grid = (dep.select("cust_pwr_id", "month")
              .join(cm.select("cust_pwr_id", "month", "n_legs"), ["cust_pwr_id", "month"], "left")
              .fillna({"n_legs": 0}))
    w4 = W.partitionBy("cust_pwr_id").orderBy("month")
    silence = (grid
        .withColumn("fwd", F.sum("n_legs").over(w4.rowsBetween(0, CONFIG["SILENCE_MONTHS"] - 1)))
        .withColumn("silent", (F.col("fwd") == 0).cast("int"))
        .groupBy("cust_pwr_id").agg(F.max("silent").alias("txn_silence")))

    labels = (dep_ids
        .join(silence, "cust_pwr_id", "left")
        .join(pat.select("cust_pwr_id",
                         (F.col("shape") == "STOPPED_BEFORE_END").cast("int").alias("series_stopped")),
              "cust_pwr_id", "left")
        .join(survivors.withColumn("balance_30pct", F.lit(1)), "cust_pwr_id", "left")
        .fillna({"txn_silence": 0, "series_stopped": 0, "balance_30pct": 0}))

    save(labels.groupBy("balance_30pct", "series_stopped", "txn_silence").count()
               .orderBy(F.desc("count")).toPandas(),
         "E4_label_agreement", "*** how much the three targets disagree ***")
    save(labels.agg(F.mean(F.col("balance_30pct").cast("double")).alias("rate_balance_30pct"),
                    F.mean(F.col("series_stopped").cast("double")).alias("rate_series_stopped"),
                    F.mean(F.col("txn_silence").cast("double")).alias("rate_txn_silence")).toPandas(),
         "E4_label_prevalence", "prevalence drives every downstream power calculation")
    # CAVEAT: txn_silence is measured over PROFILE_MONTHS only and is inflated at
    # this window width. Re-read after widening.

## E5 · Summary

In [ ]:
save(pd.DataFrame(list(R.items()), columns=["metric", "value"]), "E5_SUMMARY")
save(pd.DataFrame(TIMINGS), "E5_TIMINGS", "project the full-window run from these")

print("""
READING THE SUMMARY
-------------------
partitioned                    False => every staging read is a full scan. Raise it; it gates
                               the whole programme, not just this notebook.
legs_per_txn                   ~1.x expected. Near 2.0 across ALL topologies means the source
                               double-books before we explode, and C2 double-counts flow.
off_us_outbound_dollar_share   the half of the payment picture the graph has never had.
cpty_name_dollar_coverage      >0.70 Group A live as specified · 0.40-0.70 live on a biased
                               subset, always quote the rail split · <0.40 rail-specific.
cpty_key_dollar_coverage       account_derived dominant => stable counterparty nodes, Group B
                               dissolution metrics trustworthy. name_derived dominant => a
                               spelling change reads as a lost relationship; Group B needs a caveat.
corr_delta_netflow             >0.8 ONE ledger — reframe: counterparty IDENTITY is the new axis,
                               not amount or timing. <0.4 two systems. Monthly-average balance
                               caps this regardless, so do not read 0.6 as "half missing".
deposit_outbound_coverage      replaces the ~12% figure, over PROFILE_MONTHS only.
episodes_outbound_visible      <300 underpowered — ship Steps 1-2 and stop · 300-1000 descriptive
                               separation curves only · >1000 the brief's plan runs as written.

BEFORE WIDENING THE WINDOW
    Read E5_TIMINGS and multiply by the month count. If A3 dominates, it is the only
    stage that touches staging — everything else scales with the working tables, which
    are one to two orders of magnitude smaller.

PINNED FOR THE DEPOSIT TEAM
    monthly closed-account count, or per-account open/close dates. Converts this from
    predicting a balance-derived event to predicting an actual departure.
""")